## Find Best Linear Probes (by MSE) for 4 Models × 3 Datasets

In [1]:
import os
import json
import re
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path("../outputs")

MODEL_NAMES = [
    "Qwen3-1_7B-non-thinking",
    "Qwen3-4B-non-thinking",
    "Qwen3-8B-non-thinking",
    "Qwen3-14B-non-thinking",
]
TRAIN_DATASETS = ["triviaqa-train", "gsm8k-train", "math-train"]

def parse_hparams(probe_dir_name: str) -> dict:
    """Extract hyperparameters from the probe directory name."""
    hparams = {}
    for part in probe_dir_name.split("__"):
        if "-" in part:
            key, val = part.split("-", 1)
            hparams[key] = val
    return hparams

rows = []
for model in MODEL_NAMES:
    for dataset in TRAIN_DATASETS:
        probes_dir = OUTPUTS_DIR / f"{dataset}__{model}" / "probes"
        if not probes_dir.exists():
            continue
        for probe_dir in probes_dir.iterdir():
            if not probe_dir.name.startswith("linear__"):
                continue
            results_file = probe_dir / "layer_mean_pooled" / "results.json"
            if not results_file.exists():
                continue
            with open(results_file) as f:
                results = json.load(f)
            hparams = parse_hparams(probe_dir.name)
            rows.append({
                "model": model,
                "dataset": dataset,
                "probe_dir": str(probe_dir.name),
                "lr": hparams.get("lr", ""),
                "preprocess": hparams.get("preprocess", ""),
                "epochs": hparams.get("epochs", ""),
                "num_samples": hparams.get("num_samples", ""),
                "optimizer": hparams.get("optimizer", ""),
                "weight_decay": hparams.get("weight_decay", ""),
                "loss": hparams.get("loss", ""),
                "mse": results["evaluation"]["mse"],
                "mae": results["evaluation"]["mae"],
                "ece": results["evaluation"]["ece"],
                "pearson_r": results["evaluation"]["pearson_r"],
                "best_probe_path": str(probe_dir / "layer_mean_pooled" / "best_probe.pt"),
            })

df = pd.DataFrame(rows)
print(f"Total probe results found: {len(df)}")
print(f"Unique (model, dataset) combos: {df.groupby(['model', 'dataset']).ngroups}")
df.head()

Total probe results found: 224
Unique (model, dataset) combos: 12


,model,dataset,probe_dir,lr,preprocess,epochs,num_samples,optimizer,weight_decay,loss,mse,mae,ece,pearson_r,best_probe_path
0,Qwen3-1_7B-non-thinking,triviaqa-train,linear__loss-bce__optimizer-sgd__lr-1e-4__lr_s...,1e-4,false,100,76523,sgd,1e-2,bce,0.118522,0.229871,0.079209,0.538342,../outputs/triviaqa-train__Qwen3-1_7B-non-thin...
1,Qwen3-1_7B-non-thinking,triviaqa-train,linear__loss-bce__optimizer-sgd__lr-2e-4__lr_s...,2e-4,true,100,76523,sgd,1e-2,bce,0.107837,0.250786,0.021920,0.558004,../outputs/triviaqa-train__Qwen3-1_7B-non-thin...
2,Qwen3-1_7B-non-thinking,triviaqa-train,linear__loss-bce__optimizer-sgd__lr-1e-3__lr_s...,1e-3,false,100,76523,sgd,1e-2,bce,0.198883,0.248134,0.234723,0.477328,../outputs/triviaqa-train__Qwen3-1_7B-non-thin...
3,Qwen3-1_7B-non-thinking,triviaqa-train,linear__loss-bce__optimizer-sgd__lr-5e-4__lr_s...,5e-4,false,100,76523,sgd,1e-2,bce,0.171212,0.228854,0.203457,0.489164,../outputs/triviaqa-train__Qwen3-1_7B-non-thin...
4,Qwen3-1_7B-non-thinking,triviaqa-train,linear__loss-bce__optimizer-sgd__lr-5e-5__lr_s...,5e-5,true,100,76523,sgd,1e-2,bce,0.109306,0.251454,0.022024,0.550520,../outputs/triviaqa-train__Qwen3-1_7B-non-thin...


### All Results by (Model, Dataset) — Sorted by MSE

In [2]:
display_cols = ["model", "dataset", "optimizer", "lr", "preprocess", "epochs", "num_samples", "weight_decay", "loss", "mse", "mae", "ece", "pearson_r"]
df_sorted = df[display_cols].sort_values(["model", "dataset", "mse"])
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 40)
df_sorted

,model,dataset,optimizer,lr,preprocess,epochs,num_samples,weight_decay,loss,mse,mae,ece,pearson_r
211,Qwen3-14B-non-thinking,gsm8k-train,sgd,5e-5,false,500,7473,1e-2,bce,0.034398,0.047156,0.047155,0.036922
200,Qwen3-14B-non-thinking,gsm8k-train,sgd,1e-4,false,500,7473,1e-2,bce,0.034398,0.047157,0.047157,0.000000
204,Qwen3-14B-non-thinking,gsm8k-train,sgd,5e-4,false,500,7473,1e-2,bce,0.034398,0.047157,0.047157,0.000000
207,Qwen3-14B-non-thinking,gsm8k-train,sgd,1e-5,false,500,7473,1e-2,bce,0.035909,0.055589,0.045314,0.115032
209,Qwen3-14B-non-thinking,gsm8k-train,sgd,2e-4,true,500,7473,1e-2,bce,0.036799,0.109904,0.043443,0.182343
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Qwen3-8B-non-thinking,triviaqa-train,closed_form,,false,,76523,2,mse,0.123158,0.299817,0.054378,0.588432
94,Qwen3-8B-non-thinking,triviaqa-train,sgd,1e-3,false,100,76523,1e-2,bce,0.123772,0.229146,0.105629,0.623483
128,Qwen3-8B-non-thinking,triviaqa-train,sgd,1e-4,true,100,3736,1e-2,bce,0.124632,0.271613,0.040421,0.586607
98,Qwen3-8B-non-thinking,triviaqa-train,sgd,1e-5,true,1000,3736,1e-2,bce,0.124852,0.273909,0.041945,0.585054


### Best Linear Probe per (Model, Dataset) Combination

In [3]:
best_idx = df.groupby(["model", "dataset"])["mse"].idxmin()
df_best = df.loc[best_idx].sort_values(["model", "dataset"]).reset_index(drop=True)

best_display = df_best[["model", "dataset", "optimizer", "lr", "preprocess", "epochs", "num_samples", "weight_decay", "mse", "mae", "ece", "pearson_r"]]
best_display

,model,dataset,optimizer,lr,preprocess,epochs,num_samples,weight_decay,mse,mae,ece,pearson_r
0,Qwen3-14B-non-thinking,gsm8k-train,sgd,5e-5,false,500,7473,1e-2,0.034398,0.047156,0.047155,0.036922
1,Qwen3-14B-non-thinking,math-train,sgd,2e-4,true,500,11498,1e-2,0.037353,0.122510,0.039009,0.666158
2,Qwen3-14B-non-thinking,triviaqa-train,sgd,5e-5,true,100,76523,1e-2,0.087233,0.219165,0.033156,0.704705
3,Qwen3-1_7B-non-thinking,gsm8k-train,sgd,2e-4,true,500,7473,1e-2,0.079996,0.201606,0.019374,0.409790
4,Qwen3-1_7B-non-thinking,math-train,sgd,5e-4,true,500,11498,1e-2,0.066708,0.181005,0.023439,0.689299
5,Qwen3-1_7B-non-thinking,triviaqa-train,sgd,2e-4,true,100,76523,1e-2,0.107837,0.250786,0.021920,0.558004
6,Qwen3-4B-non-thinking,gsm8k-train,sgd,1e-4,false,500,7473,1e-2,0.045904,0.116059,0.010960,0.309357
7,Qwen3-4B-non-thinking,math-train,sgd,5e-4,true,500,11498,1e-2,0.046522,0.140409,0.027151,0.683895
8,Qwen3-4B-non-thinking,triviaqa-train,sgd,1e-4,true,100,76523,1e-2,0.111529,0.257181,0.032794,0.668706
9,Qwen3-8B-non-thinking,gsm8k-train,sgd,2e-5,false,500,7473,1e-2,0.036896,0.093483,0.007194,0.326406


### Pivot Table: Best MSE by Model × Dataset

In [6]:
pivot = df_best.pivot(index="model", columns="dataset", values="mse")
pivot = pivot[TRAIN_DATASETS].loc[MODEL_NAMES]
pivot  #.style.format("{:.6f}").highlight_min(axis=0, props="font-weight: bold; background-color: #d4edda")

dataset,triviaqa-train,gsm8k-train,math-train
model,,,
Qwen3-1_7B-non-thinking,0.107837,0.079996,0.066708
Qwen3-4B-non-thinking,0.111529,0.045904,0.046522
Qwen3-8B-non-thinking,0.107092,0.036896,0.038960
Qwen3-14B-non-thinking,0.087233,0.034398,0.037353


### Best Probe Paths (best_probe.pt)

In [7]:
for _, row in df_best.sort_values(["dataset", "model"]).iterrows():
    print(f"[{row['dataset']} | {row['model']}]")
    print(f"  {row['best_probe_path']}")
    print()

[gsm8k-train | Qwen3-14B-non-thinking]
  ../outputs/gsm8k-train__Qwen3-14B-non-thinking/probes/linear__loss-bce__optimizer-sgd__lr-5e-5__lr_scheduler-none__weight_decay-1e-2__batch_size-32__epochs-500__preprocess-false__apply_sigmoid-false__num_samples-7473/layer_mean_pooled/best_probe.pt

[gsm8k-train | Qwen3-1_7B-non-thinking]
  ../outputs/gsm8k-train__Qwen3-1_7B-non-thinking/probes/linear__loss-bce__optimizer-sgd__lr-2e-4__lr_scheduler-none__weight_decay-1e-2__batch_size-32__epochs-500__preprocess-true__apply_sigmoid-false__num_samples-7473/layer_mean_pooled/best_probe.pt

[gsm8k-train | Qwen3-4B-non-thinking]
  ../outputs/gsm8k-train__Qwen3-4B-non-thinking/probes/linear__loss-bce__optimizer-sgd__lr-1e-4__lr_scheduler-none__weight_decay-1e-2__batch_size-32__epochs-500__preprocess-false__apply_sigmoid-false__num_samples-7473/layer_mean_pooled/best_probe.pt

[gsm8k-train | Qwen3-8B-non-thinking]
  ../outputs/gsm8k-train__Qwen3-8B-non-thinking/probes/linear__loss-bce__optimizer-sgd__lr